In [ ]:
# Reference - https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/building_ai_agents/introduction_to_openai_ai_agents_sdk.ipynb
"""
    %pip install openai-agents pydantic mermaid-py nest-asyncio --upgrade
# NOTE - This is only required for Jupyter notebook environments:
    from nest_asyncio import apply
    apply()
"""

In [ ]:
# In the Multi-Source Customer Support Agent notebook, you wrote ~20 lines of boilerplate to run a single agent:

# MANUAL APPROACH (what you built in the Multi-source-Cust_Support_Agent.ipynb program )
def run_agent(user_query, max_turns=10):
    messages = [{"role": "user", "content": user_query}]
    number_of_turns = 0
    
    while number_of_turns <= max_turns:
        response = client.responses.create(
            model=MODEL, instructions=instructions,
            input=messages, tools=tools
        )
        number_of_turns += 1
        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            return response.output_text
        messages.extend(response.output)
        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)
            result = dispatch_tool_call(tool_call.name, args)
            messages.append({"type": "function_call_output", "call_id": tool_call.call_id, "output": result})
    return response.output_text
With the SDK, the same thing becomes:

# SDK APPROACH (what you'll learn in this notebook)
from agents import Agent, Runner, function_tool

agent = Agent(name="Support", instructions=instructions, tools=[search_orders, search_products])
result = Runner.run_sync(agent, "What's the status of order #1003?")
print(result.final_output)
The SDK handles the while loop, tool dispatch, argument parsing, function_call_output messages, and turn counting. You just define your agents and tools.

In [ ]:
# SET UP 
import getpass
import os
# The SDK reads OPENAI_API_KEY from the environment automatically.
# No need to instantiate an OpenAI() client.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
print("OPENAI_API_KEY is set. Ready to go.")

In [ ]:
# -----------------PART 12
# An Agent is a large language model configured with a name, instructions, and optional tools. 
# You run it with Runner.run_sync() (synchronous) or await Runner.run() (async).
from agents import Agent, Runner

agent = Agent(
    name="Haiku Writer",
    instructions="You are a creative poet. Always respond with a haiku (5-7-5 syllable pattern).",
    model="gpt-5-mini",
)

result = Runner.run_sync(agent, "Write about Python programming.")
print(result.final_output)

Indent guides my code
Libraries craft simple tools
Zen whispers, clean code
Runner.run_sync() handles the full agent loop internally. The result object contains:

result.final_output — the agent's final text response
result.new_items — all items (messages, tool calls) generated during the run
result.last_agent — the agent that produced the final output (useful with handoffs)

In [ ]:
# -----------------PART 2   Custom Tools with @function_tool
"""Reference -- old way of doing things is at Multi-source-Cust_Support_Agent.ipynb program

The OpenAI Agents SDK (openai-agents) is a lightweight framework that eliminates the boilerplate of building agentic applications. 
In the previous notebook you built a manual agent loop from scratch: a while loop, a dispatch dictionary, function_call_output messages, 
and careful bookkeeping. The SDK handles all of that for you, so you can focus on defining agents, tools, and orchestration logic.
"""
from agents import function_tool
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name to check weather for.
    """
    # In production, this would call a weather API.
    weather_data = {
        "london": "Cloudy, 14°C",
        "tokyo": "Sunny, 28°C",
        "new york": "Partly cloudy, 22°C",
    }
    return weather_data.get(city.lower(), f"No weather data available for {city}")


# The decorator auto-generates the JSON schema:
print(f"Tool name: {get_weather.name}")
print(f"Tool description: {get_weather.description}")
print(f"Parameters schema: {get_weather.params_json_schema}")

Tool name: get_weather
Tool description: Get the current weather for a city.
Parameters schema: {'properties': {'city': {'description': 'The city name to check weather for.', 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}

In [ ]:
weather_agent = Agent(
    name="Weather Assistant",
    instructions="You help users check the weather. Use the get_weather tool.",
    model="gpt-5-mini",
    tools=[get_weather],
)

result = Runner.run_sync(weather_agent, "What's the weather like in Tokyo?")
print(result.final_output)
# Tokyo: Sunny, 28°C.

In [ ]:
# ------------------Part 3: Built-in Tools - The SDK provides ready-made wrappers for OpenAI's hosted tools. Instead of writing {"type": "web_search_preview"}, you use WebSearchTool(). 
# Instead of {"type": "code_interpreter"}, you use CodeInterpreterTool(). These tools run on OpenAI's servers, so there is nothing to implement on your side.
from agents import WebSearchTool

research_agent = Agent(
    name="Web Researcher",
    instructions="You research topics using web search. Provide concise, factual answers with sources.",
    model="gpt-5-mini",
    tools=[
        WebSearchTool(search_context_size="medium"),
    ],
)

result = Runner.run_sync(research_agent, "What is the latest version of the OpenAI Agents SDK?")
print(result.final_output)

Do you mean the Python or the JavaScript/TypeScript SDK? Short answer — as of today (March 5, 2026):

- Python SDK: v0.10.4 (released Mar 3, 2026). ([github.com](https://github.com/openai/openai-agents-python))  
- JavaScript/TypeScript SDK: v0.5.4 (released Mar 5, 2026). ([github.com](https://github.com/openai/openai-agents-js/releases))

In [ ]:
# -------------------Part 4 Structured output - With the SDK, you just pass output_type=MyModel and result.final_output is already a 
# parsed Pydantic instance.
"""
In previous notebooks, getting structured JSON output required:

Defining a Pydantic model
Calling .model_json_schema() to generate the schema
Passing text={"format": {"type": "json_schema", "name": "...", "strict": True, "schema": ...}}
Parsing with Model.model_validate_json(response.output_text)
With the SDK, you just pass output_type=MyModel and result.final_output is already a parsed Pydantic instance.
"""
from pydantic import BaseModel, Field

class MovieRecommendation(BaseModel):
    """A structured movie recommendation."""
    title: str = Field(description="The movie title")
    year: int = Field(description="Release year")
    genre: str = Field(description="Primary genre")
    reason: str = Field(description="Why this movie is recommended")
    rating: float = Field(description="Rating out of 10")


recommender = Agent(
    name="Movie Recommender",
    instructions="Recommend a single movie based on the user's mood or preference.",
    model="gpt-5-mini",
    output_type=MovieRecommendation,
)

result = Runner.run_sync(recommender, "I want something mind-bending and thought-provoking.")

# result.final_output is already a MovieRecommendation instance!
movie = result.final_output
print(f"Title: {movie.title} ({movie.year})")
print(f"Genre: {movie.genre}")
print(f"Rating: {movie.rating}/10")
print(f"Reason: {movie.reason}")

Title: Coherence (2013)
Genre: Science fiction
Rating: 8.5/10
Reason: A low-key, tightly plotted sci-fi thriller about a dinner party that fractures into alternate realities. It’s mind-bending without relying on effects—focuses on identity, choice, and the unsettling consequences of parallel timelines, with an ambiguous ending that sparks discussion.
No manual JSON schema generation. No model_validate_json(). The SDK handles the entire structured output pipeline.

In [ ]:
# --------------------------Part 5 - Agents as Tool 
# Sometimes you want one agent to delegate a sub-task to another agent without transferring control of the conversation. 
# The .as_tool() method wraps an agent as a callable tool.

# This differs from handoffs (Part 6) in two ways:

# 1. The sub-agent receives generated input (not the full conversation history)
# 2. The original agent continues the conversation after the tool call returns
spanish_translator = Agent(
    name="Spanish Translator",
    instructions="Translate the input text to Spanish. Return only the translation.",
    model="gpt-5-mini",
)

french_translator = Agent(
    name="French Translator",
    instructions="Translate the input text to French. Return only the translation.",
    model="gpt-5-mini",
)

from typing import Optional

class Translations(BaseModel):
    french_text: Optional[str] = Field(None)
    spanish_text: Optional[str] = Field(None)

orchestrator = Agent(
    name="Translation Orchestrator",
    instructions=(
        "You help users translate text. Use the available translation tools. "
        "Always provide translations in all available languages."
    ),
    model="gpt-5-mini",
    output_type=Translations,
    tools=[
        spanish_translator.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate text to Spanish",
        ),
        french_translator.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate text to French",
        ),
        WebSearchTool(search_context_size="medium"),
    ],
)

changed_query = '''I want you to do some web search for me on women's dresses. Then translate your research into both languages.'''
result = Runner.run_sync(orchestrator, changed_query)
print(result.final_output)

model = result.final_output

model.spanish_text

In [ ]:
#------------------------Part 6: Handoffs
# Handoffs are the other way to build multi-agent systems. Unlike agents-as-tools, a handoff transfers control of the conversation 
# to a different agent. The new agent receives the full conversation history and takes over completely.

# Handoffs are exposed to the LLM as tools named transfer_to_<agent_name>. The model decides when to hand off based on 
# its instructions.
billing_agent = Agent(
    name="Billing Agent",
    instructions="You handle billing questions. Explain pricing plans and payment options clearly.",
    model="gpt-5-mini",
)

tech_support_agent = Agent(
    name="Tech Support Agent",
    instructions="You handle technical issues. Help users troubleshoot problems step by step.",
    model="gpt-5-mini",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are the first point of contact. Determine what the user needs and "
        "hand off to the appropriate specialist agent. "
        "For billing questions, hand off to the Billing Agent. "
        "For technical issues, hand off to the Tech Support Agent."
    ),
    model="gpt-5-mini",
    handoffs=[billing_agent, tech_support_agent],
)

# Billing question -> should hand off to Billing Agent
result = Runner.run_sync(triage_agent, "How much does the Pro plan cost?")
print(f"Handled by: {result.last_agent.name}")
print(f"Answer: {result.final_output}")

# Technical question -> should hand off to Tech Support Agent
result = Runner.run_sync(triage_agent, "My API calls are returning 429 errors. Help!")
print(f"Handled by: {result.last_agent.name}")
print(f"Answer: {result.final_output}")

Handled by: Tech Support Agent
Answer: I'm handing this over to our Tech Support Agent who can look into your account and logs. To help them diagnose faster, please paste the following (or grant access if requested):

- Exact API endpoint(s) you're calling and approximate request rate (requests/sec).
- Full HTTP response headers and body for a 429 response (especially Retry-After, x-rate-limit headers, request-id if present).
- A short code snippet or curl command that reproduces the problem.
- Time(s) (including timezone) when you saw the errors.
- Whether you're using an SDK or raw HTTPS, and which version of the SDK.
- Any recent changes (new deployments, traffic spikes, new endpoints).
- Your account/project ID or the email associated with the account (do NOT paste secrets or API keys).

In the meantime, try these immediate mitigations:
- Honor the Retry-After header and implement exponential backoff + jitter.
- Reduce request rate or batch requests if possible.
- Cache responses where appropriate.
- Check if multiple workers/servers are sharing the same API key and unintentionally exceeding limits.
- If you need higher quota, mention that to the support agent so they can escalate.

Tech Support Agent will reach out here shortly.

In [ ]:
# ------------------------Part 7: Multi-turn Conversations . To carry conversation state between turns, 
# use result.to_input_list() to convert the previous run's output into input for the next turn.
assistant = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Reply concisely.",
    model="gpt-5-mini",
)

# List concatenation: [2]  + [2] = [2, 2]

# Turn 1
result = Runner.run_sync(assistant, "What city is the Golden Gate Bridge in?")
print(f"Turn 1: {result.final_output}")

# Turn 2 — pass previous context + new question
new_input = result.to_input_list() + [{"role": "user", "content": "What state is it in?"}]
result = Runner.run_sync(assistant, new_input)
print(f"Turn 2: {result.final_output}")

# Turn 3 — continue the conversation
new_input = result.to_input_list() + [{"role": "user", "content": "What is the population of that state?"}]
result = Runner.run_sync(assistant, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: The Golden Gate Bridge is in San Francisco, California — it spans the Golden Gate strait, connecting the City of San Francisco to Marin County (near Sausalito).
Turn 2: The Golden Gate Bridge is in California, USA.
Turn 3: As of the 2020 U.S. Census, California's population was 39,538,223. More recent estimates (2021–2023) put it at about 39.2 million people (U.S. Census Bureau annual estimates). Would you like the exact latest estimate for a specific year?

In [ ]:
# -----------------------Part 8: Tracing - The trace() context manager groups multiple agent runs into a single trace that you can view in the OpenAI dashboard. 
# This is useful for debugging and understanding how your multi-agent system behaves.

from agents import trace

with trace("Translation Workflow"):
    # All Runner.run_sync calls inside this block are grouped into one trace
    result = Runner.run_sync(
        orchestrator,
        "Translate 'The weather is beautiful today' to all available languages.",
    )
    print(result.final_output)

print("\nTrace has been sent to the OpenAI dashboard.")
print("View it at: https://platform.openai.com/traces")

french_text="Il fait beau aujourd'hui." spanish_text='Hace buen tiempo hoy.'

Trace has been sent to the OpenAI dashboard.
View it at: https://platform.openai.com/traces